# Solutions · 07 · Advection: the hard case

Worked solutions to the exercises of [notebook 07](../notebooks/07_advection_the_hard_case.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, analysis, stiffness, grids, boundary as bc, operators as op
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Compute the fully discrete amplification |R(h sigma(theta))| for upwind + Euler at Courant numbers 0.5, 1.0 and 1.1 with `analysis.amplification`, and for centred + RK4 at Courant numbers 2.5 and 3.0. Confirm the stability limits by integrating the pulse.

In [2]:
dx = 1 / 200
for stencil, method, nus in (("first_upwind", "euler", (0.5, 1.0, 1.1)), ("first_central", "rk4", (2.5, 3.0))):
    for nu in nus:
        amp = np.max(analysis.amplification(stencil, method, nu * dx, dx, sign=-1.0))
        g = grids.cell_centred(0, 1, 200)
        sch = "upwind" if stencil == "first_upwind" else "centred"
        f = lambda t, u, sch=sch: op.advection(u, g, 1.0, bc.Periodic(), bc.Periodic(), t, sch)
        tab = ti.EULER if method == "euler" else ti.RK4
        s = ti.explicit_rk(f, np.exp(-((g.x - 0.3) / 0.05) ** 2), (0, 2.0), nu * dx, tab, n_out=1)
        print(f"{stencil:<13} + {method}: Courant {nu}: max amplification {amp:.4f}; max |u| after t = 2 ({s.steps} steps): {np.max(np.abs(s.y[-1])):.3g}")

first_upwind  + euler: Courant 0.5: max amplification 1.0000; max |u| after t = 2 (800 steps): 0.447
first_upwind  + euler: Courant 1.0: max amplification 1.0000; max |u| after t = 2 (400 steps): 0.998
first_upwind  + euler: Courant 1.1: max amplification 1.2000; max |u| after t = 2 (364 steps): 1e+12
first_central + rk4: Courant 2.5: max amplification 1.0000; max |u| after t = 2 (160 steps): 0.93
first_central + rk4: Courant 3.0: max amplification 1.5052; max |u| after t = 2 (133 steps): 3.33e+07


The von Neumann factor is at most 1 exactly up to the Courant limits (1 for upwind + Euler, 2.83 for centred +
RK4) and exceeds it beyond; the integrations confirm it: bounded below the limit, growing above - though the
growth needs enough steps to amplify the sawtooth from round-off, which is why the runs go to t = 2 (a run
to t = 0.5 at Courant 3 still looks innocent). The analysis and the experiment agree because the problem is
linear with periodic boundaries - the case von Neumann's method covers exactly.

## Exercise 2

Reverse the flow in the plug-flow reactor (u = -1, feed at x = 1) and solve it; then set the feed concentration to a square pulse in time and watch it exit. What happens if you (wrongly) impose a Dirichlet value at the outflow?

In [3]:
Pe, Da = 10.0, 2.0
g = grids.uniform(0, 1, 100)
def rhs_reverse(t, c, feed=lambda t: 1.0):
    left, right = bc.Neumann(0.0), bc.Robin(1.0, 1 / Pe, feed(t))           # feed at x = 1 for flow to the left: c + c_x/Pe = c_feed
    return op.advection(c, g, -1.0, left, right, t, "upwind") + op.diffusion(c, g, 1 / Pe, left, right, t) - Da * c
s = ti.bdf(rhs_reverse, np.zeros(101), (0, 4.0), 0.01, 2, band=(1, 1), n_out=1)
print(f"reversed flow: outlet (x = 0) conversion {1 - s.y[-1][0]:.4f} (same Danckwerts value as before: {1 - pr.plug_flow_reactor().exact(np.array([1.0]))[0]:.4f})")
pulse = lambda t: 1.0 if 0.5 < t < 1.0 else 0.0      # noqa: E731
s = ti.bdf(lambda t, c: rhs_reverse(t, c, pulse), np.zeros(101), (0, 3.0), 0.005, 2, band=(1, 1), n_out=6)
print("outlet concentration over time:", np.round(s.y[:, 0], 4))
def rhs_wrong(t, c):
    out = rhs_reverse(t, c)
    out[0] = 0.0                                             # (wrongly) pinning the outflow value at 0 too
    return out
s = ti.bdf(rhs_wrong, np.zeros(101), (0, 4.0), 0.01, 2, band=(1, 1), n_out=1)
print(f"with a Dirichlet value forced at the outflow: concentration next to it {s.y[-1][1]:.4f} - a spurious boundary layer, flux inconsistent with the balance")

reversed flow: outlet (x = 0) conversion 0.8198 (same Danckwerts value as before: 0.8227)


outlet concentration over time: [0.     0.     0.0317 0.1161 0.027  0.0031 0.0003]


with a Dirichlet value forced at the outflow: concentration next to it 0.0193 - a spurious boundary layer, flux inconsistent with the balance


Reversing the flow is a matter of swapping the inflow condition to the other end and the upwind direction
follows; the conversion is the same. A pulse in the feed exits after the residence time, dispersed. Forcing a
value at the outflow over-determines the hyperbolic part and produces an unphysical boundary layer in the
last cells: a hyperbolic equation takes its boundary data where the characteristics enter, and nowhere else.

## Exercise 3

**Implement it yourself:** write the second-order upwind-biased stencil (3 points upstream) and show from its modified equation that it is dispersive, not diffusive; measure the phase error of a sine wave after one period.

In [4]:
g = grids.cell_centred(0, 1, 200); dx = g.dx
def upwind2(u):                                            # second-order upwind-biased: (3 u_i - 4 u_{i-1} + u_{i-2}) / (2 dx)
    return -(3 * u - 4 * np.roll(u, 1) + np.roll(u, 2)) / (2 * dx)
me = analysis.modified_equation("upwind2", 1.0, dx)
print(f"modified equation: diffusion {me['diffusion']}, dispersion {me['dispersion']:.3e} u_xxx (and a fourth-derivative damping {me['fourth']:.2e})")
u0 = np.sin(2 * np.pi * g.x)
s = ti.dopri45(lambda t, u: upwind2(u), u0, (0, 1.0), 1e-9, 1e-11, n_out=1)
phase = np.angle(np.fft.fft(s.y[-1])[1]) - np.angle(np.fft.fft(u0)[1])
print(f"after one period the sine wave is shifted by {-phase / (2 * np.pi):.4f} wavelengths (dispersion: the predicted phase speed error is a dx^2 k^2/3 = {dx**2 * (2 * np.pi)**2 / 3:.4f})")
print(f"amplitude {np.abs(np.fft.fft(s.y[-1])[1]) / np.abs(np.fft.fft(u0)[1]):.4f} (slight damping from the fourth-derivative term)")

modified equation: diffusion 0.0, dispersion 8.333e-06 u_xxx (and a fourth-derivative damping -3.12e-08)
after one period the sine wave is shifted by 0.0003 wavelengths (dispersion: the predicted phase speed error is a dx^2 k^2/3 = 0.0003)
amplitude 1.0000 (slight damping from the fourth-derivative term)


The three-point upwind-biased stencil has no second-derivative error term - no numerical diffusion - but a
third-derivative one: it is dispersive, and a sine wave is advected at the wrong phase speed, by a Δx²k²/3
relative error that the measured phase shift reproduces. The small amplitude loss comes from the next term, a
fourth-derivative damping. Second order upwinding trades diffusion for dispersion, and limiters are needed
to tame the resulting oscillations at jumps.